# Business Admin Answer Helper — LoRA Fine-tuning (Colab T4)

Fine-tune **Qwen2.5-7B-Instruct** on the 445-pair business-administration
answer dataset from [github.com/ZHAO11451419/business-admin-answer-helper](https://github.com/ZHAO11451419/business-admin-answer-helper).

- Runtime: **T4 GPU (16GB) — free** (Runtime → Change runtime type → T4)
- Method: QLoRA (4-bit) + LoRA, ~25-40 min on T4
- Output: LoRA adapter + merged 16-bit model, optionally pushed to Hugging Face


## 1. Install dependencies
Run this cell (takes ~1-2 min).


In [ ]:
import subprocess, sys

# Install HF training stack + bitsandbytes for 4-bit QLoRA
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "-U", "transformers", "peft", "trl", "datasets", "accelerate",
    "bitsandbytes", "sentencepiece"])

# Check GPU
import torch
assert torch.cuda.is_available(), "GPU not detected — set Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))


## 2. Load the dataset
Downloads `train.jsonl` (445 pairs) from the GitHub repo and shows the type
distribution.


In [ ]:
import json, urllib.request
from collections import Counter

url = "https://raw.githubusercontent.com/ZHAO11451419/business-admin-answer-helper/main/data/train.jsonl"
req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
data = urllib.request.urlopen(req).read().decode("utf-8")

pairs = [json.loads(l) for l in data.strip().splitlines() if l.strip()]
print(f"Loaded {len(pairs)} pairs")
print(Counter(p["type"] for p in pairs))


## 3. Load base model in 4-bit (QLoRA)
Loads **Qwen/Qwen2.5-7B-Instruct** quantised to 4-bit with NF4 + double
quantisation, so it fits in 16GB VRAM.


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE = "Qwen/Qwen2.5-7B-Instruct"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE, quantization_config=bnb, device_map="auto", trust_remote_code=True)
print("Model loaded:", BASE)


## 4. Attach LoRA adapters
Only ~1-2% of parameters are trainable. Training on all 445 pairs for 3 epochs
takes roughly 25-40 minutes on a T4.


In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()


## 5. Train (SFT with Qwen chat template)
- Effective batch = 2 × 8 = 16
- Cosine LR schedule with 10% warmup
- 5% held out as validation
- Edit `EPOCHS` / `LR` below if you want.


In [ ]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

EPOCHS = 3.0
LR = 2e-4
BATCH = 2
GRAD_ACCUM = 8
MAX_LEN = 2048

def fmt(example):
    user = example["instruction"]
    if example.get("input", "").strip():
        user = user + "\n" + example["input"].strip()
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": user},
         {"role": "assistant", "content": example["output"]}],
        tokenize=False, add_generation_prompt=False)

ds = Dataset.from_list(pairs)
split = ds.train_test_split(test_size=0.05, seed=42)
train_ds, val_ds = split["train"], split["test"]

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    formatting_func=fmt,
    args=SFTConfig(
        output_dir="outputs/business-admin-answer-helper",
        max_length=MAX_LEN,
        dataset_text_field="text",
        packing=False,
        per_device_train_batch_size=BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR,
        num_train_epochs=EPOCHS,
        lr_scheduler_type="cosine",
        warmup_steps=8,
        logging_steps=5,
        save_strategy="steps",
        save_steps=200,
        eval_strategy="steps",
        eval_steps=200,
        bf16=True,
        seed=42,
        report_to=[],
    ),
)

trainer.train()
trainer.save_model("outputs/business-admin-answer-helper/adapter")
print("Adapter saved.")


## 6. Merge adapter into the base model (16-bit)
Merged model is a standalone 16-bit model you can push to Hugging Face and
serve anywhere.


In [ ]:
model = model.merge_and_unload()
merged_dir = "outputs/business-admin-answer-helper/merged"
model.save_pretrained(merged_dir)
tokenizer.save_pretrained(merged_dir)
print("Merged 16-bit model saved to", merged_dir)
print("Disk size (GB):", round(sum(os.path.getsize(os.path.join(merged_dir, f))
      for f in os.listdir(merged_dir)) / 1e9, 1))


## 7. Test the model
Ask a question in the same format the model was trained on. It should reply
with the high-mark answering format: explicit judgement, workings,
interpretation.


In [ ]:
import os

def answer(prompt, max_new=512):
    msgs = [{"role": "user", "content": prompt}]
    p = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = tokenizer(p, return_tensors="pt").to(model.device)
    out = model.generate(**inp, max_new_tokens=max_new, do_sample=False)
    return tokenizer.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)

q1 = "Melur Kapital Bhd is a Main Market company on Bursa Malaysia with a market capitalisation above RM2 billion. Its board asks whether it must prepare a sustainability report and how. Advise the board on the National Sustainability Reporting Framework (NSRF)."
print("Q:", q1[:80], "...")
print("A:", answer(q1))

q2 = "A company sells 10,000 units at RM20 each; variable cost is RM8 per unit and fixed costs are RM40,000. Compute the break-even point and margin of safety."
print("\nQ:", q2)
print("A:", answer(q2))


## 8. (Optional) Push to Hugging Face
1. Create a token at https://huggingface.co/settings/tokens (write role)
2. Run the cells below and paste the token when prompted
3. Your repo: `https://huggingface.co/<username>/business-admin-answer-helper`

> Model weights are too large for GitHub; the repo on GitHub keeps the code
> + dataset, while the trained weights live on Hugging Face.


In [ ]:
# Only run if you want to publish weights
from huggingface_hub import notebook_login, HfApi
notebook_login()

REPO = "business-admin-answer-helper"
api = HfApi()
api.create_repo(repo_id=REPO, exist_ok=True)
api.upload_folder(folder_path="outputs/business-admin-answer-helper/merged",
                  repo_id=REPO, repo_type="model")
print("Pushed to https://huggingface.co/" + REPO)
